# Notebook used to explore and filter the dataset

### Setup

In [4]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from pathlib import Path

SEED = 42
DATA_RAW = Path.cwd().parent / "data" / "raw"

### Load

In [ ]:
raw_data = pd.read_csv(DATA_RAW / "powerlifting_dataset.csv")

In [ ]:
raw_data.shape

In [ ]:
raw_data.info()

In [ ]:
raw_data.columns

In [ ]:
raw_data.isna().mean().sort_values(ascending=False)

In [ ]:
raw_data["Event"].value_counts()

In [ ]:
raw_data["Equipment"].value_counts()

### Filtering

In [34]:
filters = {
    "SBD Only": raw_data["Event"] == "SBD",
    "Raw Only": raw_data["Equipment"] == "Raw",
    "Not Disqualified": raw_data["Place"] != "DQ",
    "No doping disqualification": raw_data["Place"] != "DD",
    "Valid Squats Only": raw_data["Best3SquatKg"] > 0,
    "Valid Bench Press Only": raw_data["Best3BenchKg"] > 0,
    "Valid Deadlift Only": raw_data["Best3DeadliftKg"] > 0,
    "Sex Male or Female": raw_data["Sex"].isin(["M", "F"]),
    "Known bodyweight": raw_data["BodyweightKg"] > 0,
}

keep = pd.Series(True, index=raw_data.index)
funnel = [("Raw dataset", len(raw_data))]

for filter_name, filter_condition in filters.items():
    keep = keep & filter_condition
    funnel.append((filter_name, int(keep.sum())))

filtered_data = raw_data[keep]

funnel_df = pd.DataFrame(funnel, columns=["step", "rows_left"])
funnel_df["rows_removed"] = -funnel_df["rows_left"].diff()
funnel_df

# keep only one row per lifter
best_idx = filtered_data.groupby("Name")[
    "TotalKg"
].idxmax()  # for each athlete: the row label of their highest total
best_lifts = filtered_data.loc[best_idx]

# print(len(best_lifts), "lifters remaining after filtering and keeping only the best total per lifter")
# print(best_lifts["Name"].is_unique)

funnel_df.loc[len(funnel_df)] = ["One row per athlete", len(best_lifts), None]

# print(funnel_df)

print(filtered_data["TotalKg"].isna().sum())
print(
    best_lifts[
        ["BodyweightKg", "Age", "Best3BenchKg", "Best3SquatKg", "Best3DeadliftKg"]
    ].describe()
)  # min, max, moyenne, quartiles
print(best_lifts["Age"].isna().mean())  # share of missing ages
print(
    best_lifts["Tested"].value_counts(dropna=False)
)  # dropna=False also counts the empty cells
print(best_lifts["Sex"].value_counts())  # number of men and women
print((best_lifts["Date"].min(), best_lifts["Date"].max()))  # period covered

0
        BodyweightKg            Age   Best3BenchKg   Best3SquatKg  \
count  291787.000000  238302.000000  291787.000000  291787.000000   
mean       83.141241      27.353258     104.522964     160.969865   
std        20.901745      10.480779      42.141830      57.284170   
min        17.400000       5.000000       4.000000       4.700000   
25%        67.600000      20.000000      67.500000     115.000000   
50%        81.250000      25.000000     105.000000     160.000000   
75%        95.300000      31.500000     135.000000     200.000000   
max       260.000000      92.500000     325.000000     490.000000   

       Best3DeadliftKg  
count    291787.000000  
mean        188.959397  
std          58.608539  
min          10.000000  
25%         140.000000  
50%         190.000000  
75%         232.500000  
max         487.500000  
0.18330151788804847
Tested
Yes    229258
NaN     62529
Name: count, dtype: int64
Sex
M    200423
F     91364
Name: count, dtype: int64
('1965-09-04', '